# Observability

Este notebook registra métricas básicas de salud del pipeline.

In [0]:
from pyspark.sql import functions as F
from datetime import datetime

In [0]:
METRICS_TABLE = "byma.bronze.pipeline_metrics"

TABLES = {
    "bronze_operaciones_raw": "byma.bronze.operaciones_raw",
    "silver_operaciones_clean": "byma.silver.operaciones_clean",
    "silver_cotizaciones_historicas": "byma.silver.cotizaciones_historicas",
    "gold_fact_operaciones": "byma.gold.fact_operaciones"
}

In [0]:
metrics = []

for etapa, tabla in TABLES.items():
    count_rows = spark.table(tabla).count()

    metrics.append((
        etapa,
        tabla,
        count_rows,
        datetime.now()
    ))

df_metrics = spark.createDataFrame(
    metrics,
    ["etapa", "tabla", "filas_procesadas", "timestamp_ejecucion"]
)

display(df_metrics)

etapa,tabla,filas_procesadas,timestamp_ejecucion
bronze_operaciones_raw,byma.bronze.operaciones_raw,100000,2026-06-28T20:52:53.603Z
silver_operaciones_clean,byma.silver.operaciones_clean,100000,2026-06-28T20:52:54.121Z
silver_cotizaciones_historicas,byma.silver.cotizaciones_historicas,1008,2026-06-28T20:52:54.512Z
gold_fact_operaciones,byma.gold.fact_operaciones,100000,2026-06-28T20:52:55.282Z


In [0]:
(
    df_metrics
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(METRICS_TABLE)
)

In [0]:
display(
    spark.table(METRICS_TABLE)
    .orderBy(F.desc("timestamp_ejecucion"))
)

etapa,tabla,filas_procesadas,timestamp_ejecucion
gold_fact_operaciones,byma.gold.fact_operaciones,100000,2026-06-28T20:52:55.282Z
silver_cotizaciones_historicas,byma.silver.cotizaciones_historicas,1008,2026-06-28T20:52:54.512Z
silver_operaciones_clean,byma.silver.operaciones_clean,100000,2026-06-28T20:52:54.121Z
bronze_operaciones_raw,byma.bronze.operaciones_raw,100000,2026-06-28T20:52:53.603Z
